## Projet: Statistiques descriptives et Analyse des tests A/B.

**Realisateur: Thierno Mamoudou BALDE, Data Scientist**

* Les données sont fournies par le client: Agence des motos et tricycles;
* Nom du dataset: `ag_motos_tricycles.csv`.
  
Les résultats de ces tests A/B devront permettre d'identifier des moyens d'augmenter les revenus des conducteurs.

**Remarque :** Pour ce travail l'on va supposer que les données proviennent d'une expérience où les clients (passagers) sont sélectionnés aléatoirement et répartis en deux groupes :
1) les clients payant par carte bancaire,
2) les clients payant en espèces.

En effet sans cette hypothèse, il est impossible de tirer des conclusions causales sur l'impact du mode de paiement, variable <u>payment_type</u> sur le prix de la course, variable `fare_amount`.

L'objectif est d'appliquer les statistiques descriptives et les tests d'hypothèses en Python. Ce test A/B vise à analyser des données et à déterminer s'il existe une relation entre le mode de paiement, la variable ` payment_type` et le prix de la course, la variable `fare_amount`. 
Autrement dit, il est question de déterminer si les clients payant par carte bancaire (`credit card = 1`)  paient plus cher que ceux payant en espèces (`cash = 2`).

**Logique a suivre :**
* Importation et chargement des données : packages nécessaires pour la tache;
* Analyse exploratoire des données (AED) et tests d’hypothèses: calcul des statistiques descriptives et analyse des données; formulation des hypothèses nulle et alternative ?
* Communication des résultats aux parties prenantes: enseignements clés tirés du test A/B; recommandations commerciales;

### L'objectif fondamental de mener ce test est de comprendre s'il existe une relation entre 
### les deux variables `payment_type` et `fare_amount`

**Question de recherche pour ce projet de données:** "Voir objectif fondamental ci-haut" 

#### **<u>1. Importation et Chargement des donnees<u/>**

In [1]:
# Importation des paquets
import numpy as np
import pandas as pd
from scipy import stats

In [4]:
# Chargement du dataset
ag_data = pd.read_csv('ag_motos_tricycles.csv', index_col=0)  # Assigner la variable ag_data pour 'donnees de l'agence'

#### **<u>2. AED: Statistiques, Analyse et Hypotheses</u>**

##### 2.1 : Statistiques descriptives et Analyses

**Rappel:** la variable `payment_type` est codé ainsi qu'il suit:
* 1 = credit card
* 2 = cash
* 3 = No charge
* 4 = Dispute
* 5 = Unknown


In [6]:
# Afficer la description de ag_data pour voir
ag_data.describe(include='all')

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount
count,22699.000000,22699,22699,22699.000000,22699.000000,22699.000000,22699,22699.000000,22699.000000,22699.000000,22699.000000,22699.000000,22699.000000,22699.000000,22699.000000,22699.000000,22699.000000
unique,NaN,22687,22688,NaN,NaN,NaN,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,NaN,07/03/2017 3:45:19 PM,10/18/2017 8:07:45 PM,NaN,NaN,NaN,N,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,NaN,2,2,NaN,NaN,NaN,22600,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,1.556236,NaN,NaN,1.642319,2.913313,1.043394,NaN,162.412353,161.527997,1.336887,13.026629,0.333275,0.497445,1.835781,0.312542,0.299551,16.310502
std,0.496838,NaN,NaN,1.285231,3.653171,0.708391,NaN,66.633373,70.139691,0.496211,13.243791,0.463097,0.039465,2.800626,1.399212,0.015673,16.097295
min,1.000000,NaN,NaN,0.000000,0.000000,1.000000,NaN,1.000000,1.000000,1.000000,-120.000000,-1.000000,-0.500000,0.000000,0.000000,-0.300000,-120.300000
25%,1.000000,NaN,NaN,1.000000,0.990000,1.000000,NaN,114.000000,112.000000,1.000000,6.500000,0.000000,0.500000,0.000000,0.000000,0.300000,8.750000
50%,2.000000,NaN,NaN,1.000000,1.610000,1.000000,NaN,162.000000,162.000000,1.000000,9.500000,0.000000,0.500000,1.350000,0.000000,0.300000,11.800000
75%,2.000000,NaN,NaN,2.000000,3.060000,1.000000,NaN,233.000000,233.000000,2.000000,14.500000,0.500000,0.500000,2.450000,0.000000,0.300000,17.800000


L'on s'interessera uniquement a la relation entre le `payment_type` et le `fare_amount` payé par le client.
Une examen du prix moyen d'un trajet pour chaque mode de paiement peut donner une idée.

In [8]:
ag_data.groupby('payment_type')[['fare_amount']].mean().round(2)

,fare_amount
payment_type,
1,13.43
2,12.21
3,12.19
4,9.91


**Constats :** le prix moyen d'un trajet reglé par credit card est superieur a celui reglé par cash de `1,22 dollars`; soit près de 10%
Cependant, cette différence pourrait être due à un échantillonnage aléatoire plutôt qu'à une réelle différence de prix.
`Afin de déterminer si cette différence est statistiquement significative, un test d'hypothèse est nécessaire.`

##### 2.2 : Test d'hypoteses (Test A/B)

**Definition des hypotheses**
***Hypothèse nulle :*** Il n’y a pas de différence de prix moyen entre les clients qui paient par carte de crédit et ceux qui paient en espèces.

***Hypothèse alternative*** : Il existe une différence de prix moyen entre les clients qui paient par carte de crédit et ceux qui paient en espèces.

L’objectif de cette étape est de réaliser un test `t de Student pour échantillons indépendants`. Rappelons les étapes d’un test d’hypothèse :

* Énoncer l’hypothèse nulle et l’hypothèse alternative
* Choisir un seuil de signification
* Calculer la p-valeur
* Rejeter ou ne pas rejeter l’hypothèse nulle

**Remarque :** Dans ce cas précis, le test d’hypothèse constitue l’élément principal du test A/B.

$H_{0}$ : Il n’y a pas de différence de prix moyen entre les clients qui paient par carte de crédit et ceux qui paient en espèces.

$H_{1}$ : Il existe une différence de prix moyen entre les clients qui paient par carte de crédit et ceux qui paient en espèces.

Choisissir `5 %` comme seuil de signification.

In [11]:
carte_credit=ag_data[ag_data['payment_type']==1]['fare_amount']
especes=ag_data[ag_data['payment_type']==2]['fare_amount']
stats.ttest_ind(a=carte_credit, b=especes, equal_var=False)

TtestResult(statistic=np.float64(6.866800855655372), pvalue=np.float64(6.797387473030518e-12), df=np.float64(16675.48547403633))

**Conclusions:** Comme la valeur de p-value est significativement inférieure au seuil de signification de 5 %, rejet de l'hypothèse nulle.

Cela siginfie que: `il existe une différence statistiquement significative entre le prix moyen des courses pour les clients payant par carte bancaire et celui des clients payant en espèces.`

#### **<u>3. Communication des resultats et Recommandations</u>**

**<u>La principale conclusion commerciale</u>** est qu’inciter les clients à payer par carte bancaire peut générer davantage de revenus pour les conducteurs.

Ce projet repose sur l’hypothèse que les passagers étaient contraints de payer d’une manière ou d’une autre et qu’une fois informés de cette obligation, ils s’y conformaient systématiquement. 

Les données n’ayant pas été collectées de cette façon, il a fallu faire l’hypothèse de regrouper aléatoirement les données pour réaliser un test A/B. Cet ensemble de données ne tient pas compte d’autres explications possibles. Par exemple, les passagers n’ont peut-être pas beaucoup d’argent liquide sur eux ; il est donc plus facile de payer les courses longues/éloignées par carte bancaire. Autrement dit, il est beaucoup plus probable que le montant de la course détermine le mode de paiement, et non l’inverse.

### Felicitations!!!